In [ ]:
# Copy the prepared atlas from Drive to the Colab runtime once.
from google.colab import drive
from pathlib import Path
import shutil
import h5py
from tqdm.auto import tqdm

drive.mount('/content/drive')
src = Path('/content/drive/MyDrive/senescence_atlas_processed.h5')
dst = Path('/content/senescence_atlas_processed.h5')
partial = dst.with_suffix('.h5.partial')
# HDF5 byte size can differ across library versions after preprocessing.
# Require a completed prepared atlas and use its actual size for the copy.
with h5py.File(src, 'r') as prepared:
    assert bool(prepared.attrs.get('complete', False)), 'Preprocessing is incomplete.'
expected = src.stat().st_size

if not dst.exists() or dst.stat().st_size != expected:
    with src.open('rb') as reader, partial.open('wb') as writer:
        with tqdm(total=expected, unit='B', unit_scale=True, desc='Copying atlas') as bar:
            while chunk := reader.read(32 * 1024**2):
                writer.write(chunk)
                bar.update(len(chunk))
    assert partial.stat().st_size == expected
    partial.replace(dst)
print(f'Atlas: {dst.stat().st_size:,} bytes; free runtime disk: {shutil.disk_usage(dst.parent).free / 1e9:.1f} GB')


In [ ]:
# Full-atlas clipped-margin dictionary learning; no figures or biological scoring.
# Colab: use an A100 40-GB runtime and upload the current shared Python module.
# CALIBRATION_DIR must contain calibration.csv, experiment.npz and checkpoints/.
# The selected dictionary starts 2,000 additional updates with a fresh optimizer.
# Rerunning the cell resumes from the latest checkpoint (every RECORD_EVERY updates).
from pathlib import Path
import gc
import hashlib
import importlib
import io
import json
import math
import os
import platform
import sys
import time
import urllib.request
import uuid
import warnings

import h5py
import numpy as np
import pandas as pd
import torch
import triton
import triton.language as tl
from IPython.display import display
from tqdm.auto import tqdm

sys.dont_write_bytecode = True
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'margin_dictionary_learning.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Place margin_dictionary_learning.py in the working folder or an ancestor.')
sys.path.insert(0, str(ROOT))
import margin_dictionary_learning
importlib.reload(margin_dictionary_learning)
from margin_dictionary_learning import (
    _FISTA, _FreezingGramFISTA, _certificate, _dictionary_step, _full_precision,
)

# Editable run settings. K, lambda and q come from the completed calibration.
ATLAS_PATH = Path('/content/senescence_atlas_processed.h5')
CALIBRATION_DIR = Path('/content/drive/MyDrive/senescence_run_20pct/results')
OUTPUT_DIR = Path('/content/drive/MyDrive/senescence_fit_20pct')
TAXONOMY_PATH = None  # Optional local copy of the small, pinned annotation lookup.
N_UPDATES, LEARNING_RATE = 2000, 0.1
CODE_TOL, CODE_MAX_ITER, RECORD_EVERY = 2e-5, 20000, 50
DEVICE, BLOCK_ROWS, AUDIT_ROWS = 'cuda', 16384, 256
FIT_PATH, CSV_PATH = OUTPUT_DIR / 'atlas_fit.h5', OUTPUT_DIR / 'fit_history.csv'
CHECKPOINT_PATH, RESUME_FIT = OUTPUT_DIR / 'fit_checkpoint.npz', True
MODEL_SHA256 = hashlib.sha256((ROOT / 'margin_dictionary_learning.py').read_bytes()).hexdigest()
PROTOCOL_VERSION = 'senescence_full_atlas_v1'
TAXONOMY_URL = "https://allen-brain-cell-atlas.s3.us-west-2.amazonaws.com/metadata/Zeng-Aging-Mouse-WMB-taxonomy/20241130/cluster_mapping_pivot.csv"
TAXONOMY_MD5 = "eb988b8ad7478540a2fb97882ab9019f"
METADATA_COLUMNS = [
    "cell_label", "donor_label", "donor_age", "donor_age_category", "donor_sex",
    "donor_genotype", "region_of_interest_label", "anatomical_division_label",
    "population_sampling", "enrichment_population", "library_method", "library_label",
    "cluster_alias", "x", "y",
]


def _json(value):
    def convert(item):
        if isinstance(item, np.ndarray):
            return item.tolist()
        if isinstance(item, np.generic):
            return item.item()
        if isinstance(item, (Path, torch.device)):
            return str(item)
        if isinstance(item, bytes):
            return item.decode("utf-8")
        raise TypeError(f"Cannot encode {type(item).__name__} as JSON")
    return json.dumps(value, default=convert, sort_keys=True, separators=(",", ":"), allow_nan=False)


def _text_hash(values):
    digest = hashlib.sha256()
    for value in values:
        digest.update(str(value).encode("utf-8") + b"\0")
    return digest.hexdigest()


def _array_hash(values):
    return hashlib.sha256(np.ascontiguousarray(values).view(np.uint8)).hexdigest()


def _progress(callback, phase, done, total):
    if callback is not None:
        callback(phase, int(done), int(total))


def _source_header(handle, expected_fingerprint=None):
    if handle.attrs.get("format") != "senescence-csr-v1" or not bool(handle.attrs.get("complete", False)):
        raise ValueError("A complete senescence-csr-v1 prepared atlas is required")
    manifest = json.loads(handle.attrs["manifest_json"])
    fingerprint = hashlib.sha256(_json(manifest["identity"]).encode()).hexdigest()
    if (not manifest.get("complete", False) or manifest["fingerprint"] != fingerprint
            or handle.attrs["fingerprint"] != fingerprint
            or (expected_fingerprint is not None and fingerprint != expected_fingerprint)):
        raise ValueError("Prepared atlas fingerprint/complete state differs")
    n, p = map(int, manifest["shape"])
    nnz = int(manifest["nnz"])
    if n <= 0 or p <= 0 or p > np.iinfo(np.int16).max + 1:
        raise ValueError("Packed expression requires 1..32768 genes and at least one cell")
    if (list(handle["X"].attrs["shape"]) != [n, p]
            or handle["X"].attrs.get("encoding-type") != "csr_matrix"
            or int(handle.attrs["committed_rows"]) != n or int(handle.attrs["committed_nnz"]) != nnz):
        raise ValueError("Prepared atlas shape/commit state differs")
    for name, dtype, size in (("data", "float32", nnz), ("indices", "int32", nnz), ("indptr", "int64", n + 1)):
        dataset = handle["X"][name]
        if dataset.dtype != np.dtype(dtype) or dataset.shape != (size,):
            raise ValueError(f"Unexpected source CSR {name} shape/dtype")
    params = manifest["parameters"]
    if (params != manifest["identity"]["parameters"] or params.get("centering") is not False
            or params.get("gene_sd_scaling") is not False or params.get("output_dtype") != "float32"
            or params.get("global_scale") != "median_cell_l2_after_gene_filter"):
        raise ValueError("Unexpected prepared-atlas normalization")
    return manifest


def load_atlas_metadata(atlas_path, taxonomy_path=None, expected_fingerprint=None, progress=None):
    """Validate identifiers and source metadata before allocating the expression cache.

    Returns cell/gene axes, aligned obs DataFrame, and source/taxonomy provenance.
    Only the small pinned taxonomy CSV may be downloaded; expression stays local.
    """
    _progress(progress, "Metadata and taxonomy", 0, 1)
    if taxonomy_path is None:
        with urllib.request.urlopen(TAXONOMY_URL, timeout=60) as response:
            taxonomy_bytes = response.read(118432)
    else:
        taxonomy_bytes = Path(taxonomy_path).read_bytes()
    if len(taxonomy_bytes) != 118431 or hashlib.md5(taxonomy_bytes).hexdigest() != TAXONOMY_MD5:
        raise ValueError("Taxonomy differs from the pinned 20241130 lookup")
    taxonomy = pd.read_csv(io.BytesIO(taxonomy_bytes), dtype=str, keep_default_na=False)
    if taxonomy["cluster_alias"].duplicated().any():
        raise ValueError("Taxonomy cluster aliases are not unique")
    taxonomy = taxonomy.set_index("cluster_alias")

    # Feed the embedded CSV to pandas incrementally, hashing the same bytes once.
    class EmbeddedCSV(io.RawIOBase):
        def __init__(self, dataset):
            self.dataset, self.position, self.digest = dataset, 0, hashlib.md5()
        def readable(self):
            return True
        def readinto(self, buffer):
            count = min(len(buffer), len(self.dataset) - self.position)
            if count <= 0:
                return 0
            data = self.dataset[self.position:self.position + count].tobytes()
            buffer[:count] = data
            self.digest.update(data)
            self.position += count
            return count

    with h5py.File(atlas_path, "r") as handle:
        manifest = _source_header(handle, expected_fingerprint)
        n, p = manifest["shape"]
        cell_ids = np.asarray(handle["obs/cell_ids"].asstr(encoding="utf-8")[:], dtype=str)
        gene_ids = np.asarray(handle["var/gene_ids"].asstr(encoding="utf-8")[:], dtype=str)
        gene_symbols = np.asarray(handle["var/gene_symbols"].asstr(encoding="utf-8")[:], dtype=str)
        if (len(cell_ids) != n or len(gene_ids) != p or len(gene_symbols) != p
                or not pd.Index(cell_ids).is_unique or not pd.Index(gene_ids).is_unique
                or np.any(cell_ids == "") or np.any(gene_ids == "")
                or _text_hash(cell_ids) != manifest["identity"]["cell_order_sha256"]
                or _text_hash(gene_ids) != manifest["retained_gene_order_sha256"]):
            raise ValueError("Cell/gene identifiers or their order differ")
        embedded = EmbeddedCSV(handle["source_metadata/cell_metadata_csv"])
        source = manifest["sources"]["metadata"]
        if len(embedded.dataset) != source["bytes"]:
            raise ValueError("Embedded metadata byte count differs")
        with io.BufferedReader(embedded, buffer_size=4 * 1024**2) as stream:
            frames = list(pd.read_csv(stream, usecols=METADATA_COLUMNS, dtype=str,
                                      keep_default_na=False, chunksize=100000))
        if embedded.digest.hexdigest() != source["md5"]:
            raise ValueError("Embedded metadata failed its published MD5")
        obs = pd.concat(frames, ignore_index=True).set_index("cell_label")
        del frames
        if len(obs) != n or not obs.index.is_unique or not pd.Index(cell_ids).isin(obs.index).all():
            raise ValueError("Source metadata must contain exactly the prepared cell IDs")
        obs = obs.reindex(cell_ids)
        obs.index.name = "cell_id"
        for column in METADATA_COLUMNS:
            if column not in ("cell_label", "x", "y") and (obs[column] == "").any():
                raise ValueError(f"Missing source metadata: {column}")
        if not obs["cluster_alias"].isin(taxonomy.index).all():
            raise ValueError("Aging cluster_alias is absent from the pinned taxonomy")
        for column in taxonomy.columns:
            obs[column] = obs["cluster_alias"].map(taxonomy[column])
        for column in ("x", "y"):
            obs[column] = pd.to_numeric(obs[column].replace("", np.nan), errors="raise").astype(np.float64)
        obs["cluster_alias"] = obs["cluster_alias"].astype(np.int32)
        libraries = np.asarray(handle["obs/library_sizes"], dtype=np.float64)
        donors = np.asarray(handle["donors/labels"].asstr(encoding="utf-8")[:], dtype=str)
        codes = np.asarray(handle["obs/donor_codes"], dtype=np.int32)
        if (libraries.shape != (n,) or codes.shape != (n,)
                or not np.isfinite(libraries).all() or np.any(libraries < 0)
                or np.any(codes < 0) or np.any(codes >= len(donors))
                or _array_hash(libraries) != manifest["library_sizes_sha256"]
                or _array_hash(codes) != manifest["donor_codes_sha256"]
                or _text_hash(donors) != manifest["donor_labels_sha256"]
                or not np.array_equal(donors[codes], obs["donor_label"].to_numpy())):
            raise ValueError("Source donor alignment/library totals differ")
        obs["library_size"] = libraries
        for column in obs.select_dtypes(include=["object", "string"]).columns:
            obs[column] = obs[column].astype("category")
        result = dict(shape=(int(n), int(p)), nnz=int(manifest["nnz"]), fingerprint=manifest["fingerprint"],
                      cell_ids=cell_ids, gene_ids=gene_ids, gene_symbols=gene_symbols, obs=obs,
                      source_manifest=manifest, source_attributes=dict(handle.attrs),
                      taxonomy=dict(url=TAXONOMY_URL, md5=TAXONOMY_MD5, version="20241130"))
    _progress(progress, "Metadata and taxonomy", 1, 1)
    return result


def _check_columns(columns, pointers, p):
    if np.any(columns < 0) or np.any(columns >= p):
        raise ValueError("CSR column index is outside its axis")
    if len(columns) > 1:
        bad = columns[1:] <= columns[:-1]
        boundaries = pointers[1:-1]
        boundaries = boundaries[(boundaries > 0) & (boundaries < len(columns))]
        bad[boundaries - 1] = False
        if bad.any():
            raise ValueError("CSR columns must be strictly increasing within every row")


def load_packed_atlas(atlas_path, device, block_rows=4096, expected_fingerprint=None,
                      metadata=None, progress=None):
    """Read expression once: GPU f32 values/int16 columns, CPU int64 row pointers.

    Source hashes and canonical CSR are checked during this pass. Never downcast
    global nnz/pointers; callers may widen only each block's local column indices.
    """
    if block_rows <= 0:
        raise ValueError("block_rows must be positive")
    device = torch.device(device)
    with h5py.File(atlas_path, "r") as handle:
        manifest = _source_header(handle, expected_fingerprint)
        n, p = map(int, manifest["shape"])
        nnz = int(manifest["nnz"])
        if metadata is not None:
            if (metadata["source_manifest"]["fingerprint"] != manifest["fingerprint"]
                    or _text_hash(metadata["cell_ids"]) != manifest["identity"]["cell_order_sha256"]
                    or _text_hash(metadata["gene_ids"]) != manifest["retained_gene_order_sha256"]):
                raise ValueError("Metadata preflight and expression axes/fingerprint differ")
        for path, expected in (("obs/cell_ids", manifest["identity"]["cell_order_sha256"]),
                               ("var/gene_ids", manifest["retained_gene_order_sha256"])):
            if _text_hash(handle[path].asstr(encoding="utf-8")[:]) != expected:
                raise ValueError("Prepared atlas identifiers failed their hashes")
        pointers = np.asarray(handle["X/indptr"], dtype=np.int64)
        if (pointers[0] != 0 or pointers[-1] != nnz or np.any(np.diff(pointers) < 0)
                or _array_hash(pointers) != manifest["array_sha256"]["indptr"]):
            raise ValueError("Invalid source CSR row pointers")
        values = torch.empty(nnz, dtype=torch.float32, device=device)
        gene_indices = torch.empty(nnz, dtype=torch.int16, device=device)
        row_norm2 = torch.empty(n, dtype=torch.float64, device=device)
        norms_cpu = np.empty(n, dtype=np.float64)
        digests = {name: hashlib.sha256() for name in ("data", "indices")}
        _progress(progress, "Loading verified expression", 0, n)
        for start in range(0, n, block_rows):
            stop = min(n, start + block_rows)
            lo, hi = int(pointers[start]), int(pointers[stop])
            data = handle["X/data"][lo:hi]
            columns = handle["X/indices"][lo:hi]
            local_ptr = pointers[start:stop + 1] - lo
            if not np.isfinite(data).all() or np.any(data <= 0):
                raise ValueError("Expression entries must be finite and strictly positive")
            _check_columns(columns, local_ptr, p)
            digests["data"].update(data.view(np.uint8))
            digests["indices"].update(columns.view(np.uint8))
            values[lo:hi].copy_(torch.from_numpy(data))
            gene_indices[lo:hi].copy_(torch.from_numpy(columns.astype(np.int16)))
            block_norms = np.zeros(stop - start, dtype=np.float64)
            nonempty = np.diff(local_ptr) > 0
            if nonempty.any():
                block_norms[nonempty] = np.add.reduceat(np.square(data, dtype=np.float64), local_ptr[:-1][nonempty])
            norms_cpu[start:stop] = block_norms
            row_norm2[start:stop].copy_(torch.from_numpy(block_norms))
            _progress(progress, "Loading verified expression", stop, n)
        if any(digests[name].hexdigest() != manifest["array_sha256"][name] for name in digests):
            raise ValueError("Prepared expression failed its full-array hashes")
        median_norm = float(np.median(np.sqrt(norms_cpu)))
        if not np.isfinite(median_norm) or abs(median_norm - 1.0) > 2e-6:
            raise ValueError(f"Prepared expression median cell norm is not one: {median_norm}")
    return dict(values=values, gene_indices=gene_indices, indptr=pointers,
                row_norm2=row_norm2, shape=(n, p), fingerprint=manifest["fingerprint"])


def _write_strings(group, name, values):
    encoded = [str(value).encode("utf-8") for value in values]
    width = max((len(value) for value in encoded), default=1)
    group.create_dataset(name, data=np.asarray(encoded, dtype=f"S{max(width, 1)}"),
                         compression="gzip", compression_opts=1, shuffle=True)


def _write_obs(group, obs):
    for name in obs.columns:
        series = obs[name]
        if isinstance(series.dtype, pd.CategoricalDtype) or pd.api.types.is_string_dtype(series.dtype):
            categorical = pd.Categorical(series)
            field = group.create_group(name)
            field.attrs["encoding-type"] = "categorical"
            field.attrs["missing-code"] = -1
            field.create_dataset("codes", data=categorical.codes.astype(np.int32), compression="gzip", compression_opts=1)
            _write_strings(field, "categories", categorical.categories)
        else:
            group.create_dataset(name, data=series.to_numpy(), compression="gzip", compression_opts=1, shuffle=True)


def _read_obs(group, name):
    node = group[name]
    if isinstance(node, h5py.Group):
        codes = node["codes"][:]
        categories = node["categories"].asstr(encoding="utf-8")[:]
        if np.any(codes < 0) or np.any(codes >= len(categories)):
            raise ValueError(f"Invalid/missing output categorical codes: {name}")
        return categories[codes]
    return node[:]


def write_fit_h5(output_path, S, A, initial_S, metadata, config, provenance,
                 final_metrics, run_id, block_rows=4096, progress=None):
    """Write and read back a portable final fit, then atomically publish it.

    A has cells x programs; signed unit-norm S has programs x genes. Only one
    block of dense activity codes reaches host memory at a time. No optimizer state.
    """
    output_path = Path(output_path)
    partial = output_path.with_name(output_path.name + ".partial")
    if output_path.exists() or partial.exists():
        raise FileExistsError(f"Refusing to overwrite an existing fit/partial: {output_path}")
    if block_rows <= 0 or not str(run_id):
        raise ValueError("Positive block_rows and a run_id are required")
    n, p = len(metadata["cell_ids"]), len(metadata["gene_ids"])
    if S.ndim != 2 or A.ndim != 2 or initial_S.shape != S.shape:
        raise ValueError("Invalid S/A/initial_S dimensions")
    k = int(S.shape[0])
    if tuple(S.shape) != (k, p) or tuple(A.shape) != (n, k) or k <= 0:
        raise ValueError("S/A dimensions do not match the cell/gene axes")
    obs = metadata["obs"]
    if not np.array_equal(obs.index.to_numpy(dtype=str), metadata["cell_ids"]):
        raise ValueError("Output metadata is not in the exact cell order")
    s_cpu = S.detach().to(device="cpu", dtype=torch.float32).numpy()
    initial_cpu = initial_S.detach().to(device="cpu", dtype=torch.float32).numpy()
    if (not np.isfinite(s_cpu).all() or not np.isfinite(initial_cpu).all()
            or not np.allclose(np.linalg.norm(s_cpu.astype(np.float64), axis=1), 1, atol=2e-5, rtol=0)):
        raise ValueError("Final S must be finite with unit-norm rows; initial_S must be finite")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    json_payloads = dict(config_json=config, provenance_json=provenance,
                         final_metrics_json=final_metrics, source_manifest_json=metadata["source_manifest"],
                         source_attributes_json=metadata["source_attributes"], taxonomy_json=metadata["taxonomy"])
    json_payloads = {name: _json(value) for name, value in json_payloads.items()}
    pointers = np.zeros(n + 1, dtype=np.int64)
    expected_hashes = {name: hashlib.sha256() for name in ("data", "indices")}
    with h5py.File(partial, "x") as handle:
        handle.attrs.update(format="senescence-fit-v1", schema_version=1, complete=False,
                            run_id=str(run_id), shape=np.asarray([n, k, p], dtype=np.int64),
                            axes="A: cell,program; S and initial_S: program,gene",
                            source_fingerprint=metadata["source_manifest"]["fingerprint"])
        for name, value in json_payloads.items():
            handle.attrs[name] = value
        for name, data in (("S", s_cpu), ("initial_S", initial_cpu)):
            dataset = handle.create_dataset(name, data=data, compression="gzip", compression_opts=1, shuffle=True)
            dataset.attrs.update(axes="program,gene", signed=True)
        cells, genes = handle.create_group("obs"), handle.create_group("var")
        _write_strings(cells, "cell_ids", metadata["cell_ids"])
        _write_strings(genes, "gene_ids", metadata["gene_ids"])
        _write_strings(genes, "gene_symbols", metadata["gene_symbols"])
        _write_obs(cells, obs)
        cells.attrs["x_y_description"] = "Published source embedding; not an activity-code UMAP"
        cells.attrs["library_size_description"] = "Original raw all-gene cell library total before filtering"
        activity = handle.create_group("A")
        activity.attrs.update({"encoding-type": "csr_matrix", "shape": np.asarray([n, k], dtype=np.int64), "axes": "cell,program"})
        for name, dtype in (("data", "float32"), ("indices", "int32")):
            activity.create_dataset(name, shape=(0,), maxshape=(None,), dtype=dtype,
                                    chunks=(1048576,), compression="gzip", compression_opts=1, shuffle=True)
        _progress(progress, "Writing final activity codes", 0, n)
        for start in range(0, n, block_rows):
            stop = min(n, start + block_rows)
            block = A[start:stop].detach().to(device="cpu", dtype=torch.float32).numpy()
            if not np.isfinite(block).all() or np.any(block < 0):
                raise ValueError("Activity codes must be finite and nonnegative")
            rows, columns = np.nonzero(block)
            data = block[rows, columns]
            columns = columns.astype(np.int32)
            counts = np.bincount(rows, minlength=stop - start)
            pointers[start + 1:stop + 1] = pointers[start] + np.cumsum(counts, dtype=np.int64)
            lo, hi = int(pointers[start]), int(pointers[stop])
            for name, array in (("data", data), ("indices", columns)):
                activity[name].resize((hi,))
                activity[name][lo:hi] = array
                expected_hashes[name].update(array.view(np.uint8))
            _progress(progress, "Writing final activity codes", stop, n)
        activity.create_dataset("indptr", data=pointers, compression="gzip", compression_opts=1, shuffle=True)
        hashes = {name: digest.hexdigest() for name, digest in expected_hashes.items()}
        hashes.update(indptr=_array_hash(pointers), S=_array_hash(s_cpu), initial_S=_array_hash(initial_cpu))
        handle.attrs["array_sha256_json"] = _json(hashes)
        handle.flush()

    # Read the actual portable file, including decoded metadata and all CSR values.
    with h5py.File(partial, "r+") as handle:
        for path, expected in (("obs/cell_ids", metadata["cell_ids"]), ("var/gene_ids", metadata["gene_ids"]),
                               ("var/gene_symbols", metadata["gene_symbols"])):
            if not np.array_equal(handle[path].asstr(encoding="utf-8")[:], expected):
                raise ValueError(f"Output identifier readback differs: {path}")
        for name in obs.columns:
            actual, expected = _read_obs(handle["obs"], name), obs[name].to_numpy()
            equal = (np.array_equal(actual, expected, equal_nan=True)
                     if np.issubdtype(actual.dtype, np.number) else np.array_equal(actual, expected))
            if not equal:
                raise ValueError(f"Output metadata readback differs: {name}")
        for name, expected in (("S", s_cpu), ("initial_S", initial_cpu)):
            actual = handle[name][:]
            if actual.dtype != np.dtype("float32") or not np.array_equal(actual, expected) or _array_hash(actual) != hashes[name]:
                raise ValueError(f"Output dictionary readback differs: {name}")
        if not np.allclose(np.linalg.norm(handle["S"][:].astype(np.float64), axis=1), 1, atol=2e-5, rtol=0):
            raise ValueError("Output S row norms differ")
        actual_ptr = handle["A/indptr"][:]
        if (actual_ptr.dtype != np.dtype("int64") or not np.array_equal(actual_ptr, pointers)
                or len(actual_ptr) != n + 1 or actual_ptr[0] != 0 or np.any(np.diff(actual_ptr) < 0)
                or actual_ptr[-1] != len(handle["A/data"]) or actual_ptr[-1] != len(handle["A/indices"])):
            raise ValueError("Output activity row pointers differ")
        read_hashes = {name: hashlib.sha256() for name in ("data", "indices")}
        _progress(progress, "Verifying portable final fit", 0, n)
        for start in range(0, n, block_rows):
            stop = min(n, start + block_rows)
            lo, hi = int(actual_ptr[start]), int(actual_ptr[stop])
            data, columns = handle["A/data"][lo:hi], handle["A/indices"][lo:hi]
            if (data.dtype != np.dtype("float32") or columns.dtype != np.dtype("int32")
                    or not np.isfinite(data).all() or np.any(data <= 0)):
                raise ValueError("Invalid output activity entries/dtypes")
            _check_columns(columns, actual_ptr[start:stop + 1] - lo, k)
            read_hashes["data"].update(data.view(np.uint8))
            read_hashes["indices"].update(columns.view(np.uint8))
            _progress(progress, "Verifying portable final fit", stop, n)
        if any(digest.hexdigest() != hashes[name] for name, digest in read_hashes.items()):
            raise ValueError("Output activity values/columns failed readback hashes")
        for name, value in json_payloads.items():
            if handle.attrs[name] != value:
                raise ValueError(f"Output provenance readback differs: {name}")
        handle.attrs["complete"] = True
        handle.attrs["verified_readback"] = True
        handle.flush()
    os.replace(partial, output_path)
    return dict(path=str(output_path), shape=[n, k, p], activity_nnz=int(pointers[-1]),
                run_id=str(run_id), complete=True, verified_readback=True, array_sha256=hashes)


def selected_dictionary(folder, metadata):
    """Use the completed K-stage selection, with its exact learned dictionary."""
    table = pd.read_csv(folder / 'calibration.csv')
    selected = table['selected'].astype(str).str.lower().isin(['true', '1'])
    chosen = table.loc[table.stage.eq('K') & table.status.eq('complete') & selected]
    if len(chosen) != 1:
        raise ValueError('calibration.csv must contain one completed, selected K-stage fit.')
    row = chosen.iloc[0].to_dict()
    k, alpha, q = int(row['k']), float(row['lambda']), float(row['q'])
    if k != row['k'] or k < 1 or not np.isfinite(alpha) or alpha <= 0 or not 0 <= q <= 1:
        raise ValueError('Invalid selected K, lambda or q.')
    key = f'k{k}_lambda{alpha:.12g}_q{q:.12g}'
    if row['fit_id'] != key:
        raise ValueError('The selected fit identifier disagrees with its parameters.')
    with np.load(folder / 'experiment.npz', allow_pickle=False) as experiment:
        experiment_info = json.loads(str(experiment['metadata_json']))
        if (experiment_info['experiment_id'] != row['experiment_id']
                or experiment_info['identity']['atlas_fingerprint'] != metadata['fingerprint']
                or not np.array_equal(experiment['gene_ids'], metadata['gene_ids'])):
            raise ValueError('Calibration and atlas have different identities or gene ordering.')
    checkpoint = folder / 'checkpoints' / f"{row['experiment_id'][:16]}_{key}.npz"
    with np.load(checkpoint, allow_pickle=False) as saved:
        state = json.loads(str(saved['metadata_json']))
        initial = saved['S'].copy()  # Training codes and optimizer moments are not resumed.
    last = state['trace'][-1]
    if (state['experiment_id'] != row['experiment_id'] or state['fit_id'] != key
            or state['iteration'] != row['iteration'] or last['status'] != 'complete'
            or not last['training_audited']
            or not math.isclose(last['prediction_error'], row['prediction_error'], rel_tol=1e-10)):
        raise ValueError('The selected checkpoint and completed calibration record disagree.')
    if (initial.dtype != np.float32 or initial.shape != (k, metadata['shape'][1])
            or not np.isfinite(initial).all()
            or not np.allclose(np.linalg.norm(initial, axis=1), 1, atol=2e-5, rtol=0)):
        raise ValueError('The selected dictionary must have finite, unit-norm float32 rows.')
    # CSV blanks in optional diagnostics are JSON null, not nonstandard NaN.
    record = {name: None if pd.isna(value) else value for name, value in row.items()}
    provenance = dict(calibration_record=record, calibration_experiment=experiment_info,
                      calibration_checkpoint=str(checkpoint), initialization='selected_calibration_dictionary',
                      calibration_csv_sha256=hashlib.sha256((folder / 'calibration.csv').read_bytes()).hexdigest())
    return initial, alpha, q, provenance


def expression_block(atlas, start, stop):
    """Unpack local int32 columns; global offsets remain int64."""
    pointers = atlas['indptr']
    first, last = int(pointers[start]), int(pointers[stop])
    if last - first >= 2**31:
        raise ValueError('Reduce BLOCK_ROWS: a local sparse block exceeds int32 indexing.')
    ptr = np.asarray(pointers[start:stop + 1] - first, dtype=np.int32)
    values = atlas['values'][first:last]
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='Sparse CSR tensor support is in beta state')
        return torch.sparse_csr_tensor(
            torch.as_tensor(ptr, device=values.device),
            atlas['gene_indices'][first:last].to(torch.int32), values,
            size=(stop - start, atlas['shape'][1]), check_invariants=False)


def clip_global_margins(margins, quantile, block_rows):
    """One exact positive-margin quantile; overwrite margins with weights."""
    count = int((margins > 0).sum().item())
    if not count:
        margins.zero_()
        return 0.0
    positive = margins.new_empty(count)
    offset = 0
    # Bounded selection avoids constructing full-atlas nonzero index arrays.
    for start in range(0, len(margins), block_rows):
        block = margins[start:start + block_rows]
        part = torch.masked_select(block, block > 0)
        positive[offset:offset + len(part)].copy_(part)
        offset += len(part)
    rank = float(quantile) * (count - 1)
    low, high = math.floor(rank), math.ceil(rank)
    cap = positive.kthvalue(low + 1).values
    if high != low:
        cap = torch.lerp(cap, positive.kthvalue(high + 1).values, rank - low)
    margins.clamp_min_(0).clamp_max_(cap)
    return float(cap.item())


@triton.jit
def _fista_step_kernel(yg_ptr, cross_ptr, y_ptr, a_ptr, lip_ptr, beta_ptr, alpha, n,
                       BLOCK: tl.constexpr):
    i = tl.program_id(0).to(tl.int64) * BLOCK + tl.arange(0, BLOCK)
    mask = i < n
    yg, cross = tl.load(yg_ptr + i, mask=mask), tl.load(cross_ptr + i, mask=mask)
    y, a = tl.load(y_ptr + i, mask=mask), tl.load(a_ptr + i, mask=mask)
    t = y + tl.div_rn(cross - yg - alpha, tl.load(lip_ptr))
    a_next = tl.where(t <= 0, 0.0, t)
    tl.store(a_ptr + i, a_next, mask=mask)
    tl.store(y_ptr + i, a_next + tl.load(beta_ptr) * (a_next - a), mask=mask)


class FusedFreezingGramFISTA(_FreezingGramFISTA):
    """_FreezingGramFISTA with the elementwise part of each step in one kernel."""

    def _step(self, a, y, cross, beta):
        _fista_step_kernel[(triton.cdiv(a.numel(), 1024),)](
            y @ self.gram, cross, y, a, self.lip, beta, self.alpha, a.numel(),
            BLOCK=1024, enable_fp_fusion=False)
        return a, y


@torch.no_grad()
def fused_step_matches():
    """Check that the fused step reproduces the PyTorch step bit for bit."""
    generator = torch.Generator(device=DEVICE).manual_seed(0)
    s = torch.randn(200, 512, device=DEVICE, generator=generator)
    s /= s.norm(dim=1, keepdim=True)
    a = torch.rand(3001, 200, device=DEVICE, generator=generator)
    cross = torch.randn(3001, 200, device=DEVICE, generator=generator)
    norm2 = cross.double().square().sum(1)
    results = []
    with _full_precision():
        for solver_class in (_FreezingGramFISTA, FusedFreezingGramFISTA):
            solver = solver_class(s, a, 0.05, CODE_TOL, 20, cross=cross, norm2=norm2)
            codes, y = a.clone(), a.clone()
            for j in range(20):
                codes, y = solver._step(codes, y, cross, solver.schedule[j])
            results.append((codes, y))
    return all(torch.equal(x, z) for x, z in zip(*results))


def code_atlas(atlas, s, a, margins, alpha, solver, buffers, block_rows, tol, max_iter, bar=None):
    """Code every cell at the same S, reusing one fixed-size solver."""
    initial, cross, norms = buffers
    n = len(a)
    totals = dict(residual_squared=0.0, relative_gap=0.0, relative_margin_error=0.0,
                  minimum_relative_gap=math.inf, minimum_relative_residual_norm2=math.inf,
                  mean_inner_steps=0.0, max_inner_steps=0, mean_row_steps=0.0)
    steps = []
    for start in range(0, n, block_rows):
        stop = min(start + block_rows, n)
        count = stop - start
        x = expression_block(atlas, start, stop)
        initial.zero_()
        cross.zero_()
        norms.zero_()
        initial[:count].copy_(a[start:stop])
        cross[:count].copy_(torch.sparse.mm(x, s.T.contiguous()))
        norms[:count].copy_(atlas['row_norm2'][start:stop])
        del x
        if solver is None:
            solver = TRAINING_SOLVER(s, initial, alpha, tol, max_iter, cross=cross, norm2=norms)
        coded, correlation, residual2, metrics = solver.solve(s, initial, cross=cross, norm2=norms)
        a[start:stop].copy_(coded[:count])
        margins[start:stop].copy_(coded[:count] + correlation[:count] - alpha)
        totals['residual_squared'] += residual2
        for name in ('relative_gap', 'relative_margin_error'):
            totals[name] = max(totals[name], metrics[name])
        for name in ('minimum_relative_gap', 'minimum_relative_residual_norm2'):
            totals[name] = min(totals[name], metrics[name])
        totals['mean_inner_steps'] += metrics['inner_steps'] * count / n
        totals['mean_row_steps'] += metrics['row_steps'] * len(initial) / n
        totals['max_inner_steps'] = max(totals['max_inner_steps'], metrics['inner_steps'])
        steps.append(metrics['inner_steps'])
        if bar is not None:
            bar.update(count)
    return solver, totals, steps


def atlas_gradient(atlas, s, a, weights, block_rows):
    """Accumulate the complete gradient before taking one dictionary step."""
    h = s.new_zeros((len(s), len(s)), dtype=torch.float64)
    b = s.new_zeros((s.shape[1], len(s)), dtype=torch.float64)
    for start in range(0, len(a), block_rows):
        stop = min(start + block_rows, len(a))
        x = expression_block(atlas, start, stop)
        # PyTorch converts CSC to CSR internally; make the bounded temporary explicit.
        xt = x.transpose(0, 1).to_sparse_csr()
        w = weights[start:stop]
        h.add_((w.T @ a[start:stop]).double())
        b.add_(torch.sparse.mm(xt, w).double())
        del x, xt
    return ((h @ s.double() - b.T) / (len(a) * s.shape[1])).float()


def audit_atlas(atlas, s, a, alpha, steps, block_rows, audit_rows, tol, max_iter, bar=None):
    """Direct residual certificates; any refined codes are the codes exported."""
    totals = dict(residual_squared=0.0, relative_gap=0.0, relative_margin_error=0.0,
                  minimum_relative_gap=math.inf, audit_max_refinement_steps=0)
    for block, start in enumerate(range(0, len(a), block_rows)):
        for first in range(start, min(start + block_rows, len(a)), audit_rows):
            last = min(first + audit_rows, start + block_rows, len(a))
            x = expression_block(atlas, first, last).to_dense()
            ab = a[first:last]
            scale = (0.5 * x.double().square().sum(1)).clamp_min(1e-12)
            values, residual, _ = _certificate(x, s, ab, alpha, scale)
            absolute, relative, gap, minimum = values.cpu().tolist()
            if not (all(math.isfinite(v) for v in (absolute, relative, gap, minimum))
                    and relative <= tol and gap <= tol and minimum >= -tol):
                remaining = max_iter - steps[block]
                if remaining <= 0:
                    raise RuntimeError('Final residual audit needs more coding iterations; no result was marked complete.')
                refiner = _FISTA(x, s, ab, alpha, tol, remaining)
                refined, residual, _, metrics = refiner.solve(s, ab)
                ab.copy_(refined)
                relative, gap, minimum = (metrics[name] for name in
                                          ('relative_margin_error', 'relative_gap', 'minimum_relative_gap'))
                totals['audit_max_refinement_steps'] = max(totals['audit_max_refinement_steps'], metrics['inner_steps'])
                del refiner
            totals['residual_squared'] += residual.double().square().sum().item()
            totals['relative_gap'] = max(totals['relative_gap'], gap)
            totals['relative_margin_error'] = max(totals['relative_margin_error'], relative)
            totals['minimum_relative_gap'] = min(totals['minimum_relative_gap'], minimum)
            if bar is not None:
                bar.update(last - first)
    return totals


def check_memory(metadata, k, block_rows, device):
    """Budget packed expression, full codes/margins and bounded sparse temporaries."""
    if torch.device(device).type != 'cuda':
        return
    gc.collect()
    torch.cuda.empty_cache()
    n, p = metadata['shape']
    with h5py.File(ATLAS_PATH, 'r') as source:
        pointers = source['X/indptr'][:]
    ends = np.r_[np.arange(0, n, block_rows), n]
    largest = int(np.diff(pointers[ends]).max())
    persistent = 6 * metadata['nnz'] + 8 * n + 8 * n * k + 20 * k * p
    temporary = max(80 * largest, 4 * n * k + 16 * block_rows * k)
    estimate = persistent + temporary + 128 * block_rows * k + 2 * 2**30
    free, _ = torch.cuda.mem_get_info(torch.device(device))
    if estimate > free:
        raise MemoryError(f'Estimated GPU requirement {estimate / 2**30:.1f} GiB; '
                          f'available {free / 2**30:.1f} GiB. Use a free A100 40-GB runtime '
                          'and, if necessary, reduce BLOCK_ROWS.')


def synchronize():
    if torch.device(DEVICE).type == 'cuda':
        torch.cuda.synchronize(torch.device(DEVICE))


def save_npz(path, **arrays):
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('wb') as handle:
        np.savez(handle, **arrays)
    temporary.replace(path)


def load_checkpoint():
    """Latest saved fit state, or None when starting fresh."""
    if not (RESUME_FIT and CHECKPOINT_PATH.exists()):
        return None
    with np.load(CHECKPOINT_PATH, allow_pickle=False) as saved:
        state = {name: saved[name] for name in ('S', 'A', 'margins', 'first', 'second')}
        state['metadata'] = json.loads(str(saved['metadata_json']))
    return state


def save_history(rows):
    temporary = CSV_PATH.with_suffix('.csv.partial')
    pd.DataFrame(rows).to_csv(temporary, index=False)
    temporary.replace(CSV_PATH)


def fit_atlas(atlas, initial_s, alpha, q, run_id, bar, identity, state=None):
    """A fixed number of full-batch updates; a fresh optimizer at the selected S."""
    n, p = atlas['shape']
    if state is None:
        s = torch.as_tensor(initial_s, device=DEVICE).clone()
        a = s.new_zeros((n, len(s)))
        margins = torch.empty_like(a)
        first, second = torch.zeros_like(s), torch.zeros_like(s[:, :1])
        reference_s, reference_a = initial_s.copy(), np.empty((n, len(s)), dtype=np.float32)
        history, steps, done, elapsed = [], None, 0, 0.0
    else:
        s, a, margins, first, second = (torch.as_tensor(state[name], device=DEVICE).clone()
                                        for name in ('S', 'A', 'margins', 'first', 'second'))
        reference_s, reference_a = state['S'], state['A']
        history, steps, done, elapsed = (state['metadata'][name] for name in
                                         ('history', 'steps', 'iteration', 'elapsed_seconds'))
    buffers = (s.new_zeros((BLOCK_ROWS, len(s))), s.new_zeros((BLOCK_ROWS, len(s))),
               s.new_zeros(BLOCK_ROWS, dtype=torch.float64))
    durations, inner_counts, row_counts = [], [], []
    start_time = time.perf_counter() - elapsed
    xnorm = atlas['row_norm2'].sum().item()

    def record(iteration, metrics, audited=False):
        nonlocal reference_s
        ss = s.cpu().numpy()
        cosine = np.einsum('ij,ij->i', ss.astype(float), reference_s.astype(float))
        cosine /= np.linalg.norm(ss, axis=1) * np.linalg.norm(reference_s, axis=1)
        total_a = change2 = old2 = 0.0
        counts = np.zeros(len(s), dtype=np.int64)
        for lo in range(0, n, BLOCK_ROWS):
            hi = min(lo + BLOCK_ROWS, n)
            ab = a[lo:hi].cpu().numpy()
            counts += (ab > 0).sum(axis=0)
            total_a += ab.sum(dtype=np.float64)
            if iteration:
                change2 += np.square(ab.astype(float) - reference_a[lo:hi]).sum()
                old2 += np.square(reference_a[lo:hi], dtype=np.float64).sum()
            reference_a[lo:hi] = ab
        row = dict(run_id=run_id, iteration=iteration, k=len(s), **{'lambda': alpha}, q=q,
                   n_cells=n, n_genes=p, learning_rate=LEARNING_RATE,
                   training_objective=(0.5 * metrics['residual_squared'] + alpha * total_a) / n,
                   relative_reconstruction_error=metrics['residual_squared'] / xnorm,
                   mean_active_atoms=float(counts.sum() / n), used_atoms=int((counts > 0).sum()),
                   mean_atom_cosine_previous=float(np.clip(cosine, -1, 1).mean()) if iteration else None,
                   min_atom_cosine_previous=float(np.clip(cosine, -1, 1).min()) if iteration else None,
                   relative_activity_change=math.sqrt(change2 / max(old2, 1e-24)) if iteration else None,
                   mean_inner_steps=float(np.mean(inner_counts)) if inner_counts else metrics['mean_inner_steps'],
                   mean_row_steps=float(np.mean(row_counts)) if row_counts else metrics['mean_row_steps'],
                   max_inner_steps=metrics['max_inner_steps'], relative_gap=metrics['relative_gap'],
                   relative_margin_error=metrics['relative_margin_error'],
                   minimum_relative_gap=metrics['minimum_relative_gap'],
                   minimum_relative_residual_norm2=metrics['minimum_relative_residual_norm2'],
                   audit_max_refinement_steps=metrics.get('audit_max_refinement_steps', 0),
                   seconds_per_update=float(np.mean(durations)) if durations else 0.0,
                   elapsed_seconds=time.perf_counter() - start_time,
                   peak_gpu_gib=torch.cuda.max_memory_allocated() / 2**30 if s.is_cuda else 0.0,
                   audited=audited, status='audited' if audited else 'checkpoint')
        history.append(row)
        save_history(history)
        reference_s = ss.copy()
        durations.clear()
        inner_counts.clear()
        row_counts.clear()
        return row

    def checkpoint(iteration):
        # Called right after record(), so the reference arrays hold the current S and A.
        metadata = dict(identity=identity, run_id=run_id, iteration=iteration, steps=steps,
                        elapsed_seconds=time.perf_counter() - start_time, history=history)
        save_npz(CHECKPOINT_PATH, S=reference_s, A=reference_a, margins=margins.cpu().numpy(),
                 first=first.cpu().numpy(), second=second.cpu().numpy(),
                 metadata_json=np.array(json.dumps(metadata)))

    with torch.no_grad(), _full_precision():
        solver = None
        if state is None:
            bar.reset(total=n)
            bar.set_description('Initial coding')
            solver, metrics, steps = code_atlas(atlas, s, a, margins, alpha, None, buffers,
                                               BLOCK_ROWS, CODE_TOL, CODE_MAX_ITER, bar)
            initial_coding_seconds = time.perf_counter() - start_time
            row = record(0, metrics)
            row['initial_coding_seconds'] = initial_coding_seconds
            checkpoint(0)
        else:
            row = history[-1]
        bar.reset(total=N_UPDATES)
        bar.set_description(f'Atlas fit: K={len(s)}, lambda={alpha:g}, q={q:g}')
        bar.update(done)
        for iteration in range(done + 1, N_UPDATES + 1):
            synchronize()
            tic = time.perf_counter()
            cap = clip_global_margins(margins, q, BLOCK_ROWS)
            gradient = atlas_gradient(atlas, s, a, margins, BLOCK_ROWS)
            s = _dictionary_step(s, None, None, first, second, iteration, LEARNING_RATE, gradient=gradient)
            solver, metrics, steps = code_atlas(atlas, s, a, margins, alpha, solver, buffers,
                                               BLOCK_ROWS, CODE_TOL, CODE_MAX_ITER)
            synchronize()
            durations.append(time.perf_counter() - tic)
            inner_counts.append(metrics['mean_inner_steps'])
            row_counts.append(metrics['mean_row_steps'])
            bar.update(1)
            if iteration % RECORD_EVERY == 0 and iteration != N_UPDATES:
                row = record(iteration, metrics)
                checkpoint(iteration)
            cosine = '--' if row['mean_atom_cosine_previous'] is None else f"{row['mean_atom_cosine_previous']:.4f}"
            bar.set_postfix_str(f"inner={metrics['mean_inner_steps']:.0f} | "
                                f"train loss@{row['iteration']}={row['training_objective']:.5f} | cos={cosine}")
        # The last coding pass already uses the final S. Verify with explicit residuals.
        del solver, margins, buffers, gradient, first, second
        bar.reset(total=n)
        bar.set_description('Final residual audit')
        audit_start = time.perf_counter()
        audited = audit_atlas(atlas, s, a, alpha, steps, BLOCK_ROWS, AUDIT_ROWS,
                              CODE_TOL, CODE_MAX_ITER, bar)
        metrics.update(audited)
        row = record(N_UPDATES, metrics, audited=True)
        row['final_audit_seconds'] = time.perf_counter() - audit_start
        row['last_update_weight_cap'] = cap
        save_history(history)
    return s, a, metrics, history


def run():
    global TRAINING_SOLVER
    if (N_UPDATES < 1 or CODE_MAX_ITER < 1 or RECORD_EVERY < 1 or BLOCK_ROWS < 1
            or AUDIT_ROWS < 1 or not math.isfinite(LEARNING_RATE) or LEARNING_RATE <= 0
            or not math.isfinite(CODE_TOL) or CODE_TOL <= 0):
        raise ValueError('Update counts, block sizes, learning rate and tolerance must be positive.')
    if torch.device(DEVICE).type == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('Select an A100 GPU runtime before running the full atlas.')
    if FIT_PATH.exists():
        raise FileExistsError(f'The fit is already complete: {FIT_PATH}')
    state = load_checkpoint()
    if state is None:
        for path in (CSV_PATH, CHECKPOINT_PATH, FIT_PATH.with_name(FIT_PATH.name + '.partial'),
                     CSV_PATH.with_suffix('.csv.partial')):
            if path.exists():
                raise FileExistsError(f'Use a fresh OUTPUT_DIR; an existing result will not be overwritten: {path}')
    else:
        FIT_PATH.with_name(FIT_PATH.name + '.partial').unlink(missing_ok=True)
        print(f"Resuming from update {state['metadata']['iteration']} of {N_UPDATES}.")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    run_id = uuid.uuid4().hex if state is None else state['metadata']['run_id']
    resumed_seconds = 0.0 if state is None else state['metadata']['elapsed_seconds']
    if torch.device(DEVICE).type == 'cuda':
        torch.cuda.reset_peak_memory_stats(torch.device(DEVICE))
    with tqdm(total=1, desc='Atlas metadata', dynamic_ncols=True, leave=False) as bar:
        def progress(phase, done, total):
            if bar.desc != phase or bar.total != total:
                bar.set_postfix_str('', refresh=False)
                bar.reset(total=total)
                bar.set_description(phase)
            bar.update(done - bar.n)

        metadata = load_atlas_metadata(ATLAS_PATH, TAXONOMY_PATH, progress=progress)
        initial_s, alpha, q, provenance = selected_dictionary(CALIBRATION_DIR, metadata)
        check_memory(metadata, len(initial_s), BLOCK_ROWS, DEVICE)
        fused = torch.device(DEVICE).type == 'cuda' and fused_step_matches()
        TRAINING_SOLVER = FusedFreezingGramFISTA if fused else _FreezingGramFISTA
        print(f'Coding solver: {TRAINING_SOLVER.__name__}.')
        config = dict(protocol=PROTOCOL_VERSION, K=len(initial_s), **{'lambda': alpha}, q=q,
                      n_updates=N_UPDATES, learning_rate=LEARNING_RATE, code_tol=CODE_TOL,
                      code_max_iter=CODE_MAX_ITER, record_every=RECORD_EVERY, device=DEVICE,
                      block_rows=BLOCK_ROWS, audit_rows=AUDIT_ROWS,
                      atlas_path=str(ATLAS_PATH), calibration_dir=str(CALIBRATION_DIR),
                      output_dir=str(OUTPUT_DIR), fresh_optimizer=True,
                      coding_solver=TRAINING_SOLVER.__name__,
                      numerical_precision='float32 products; float64 certificate/reduction accumulation; TF32 disabled')
        provenance.update(model_sha256=MODEL_SHA256, software=dict(
            python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
            h5py=h5py.__version__, torch=torch.__version__, cuda=torch.version.cuda),
            gpu=torch.cuda.get_device_name(torch.device(DEVICE)) if torch.device(DEVICE).type == 'cuda' else None)
        # Check serializable provenance before the expensive fit.
        _json(config)
        _json(provenance)
        identity = _json(dict(config, coding_solver=None, model_sha256=MODEL_SHA256,
                              calibration=provenance['calibration_record']))
        if state is not None and state['metadata']['identity'] != identity:
            raise ValueError('The checkpoint belongs to a different fit; use a fresh OUTPUT_DIR.')
        atlas = load_packed_atlas(ATLAS_PATH, DEVICE, BLOCK_ROWS, metadata=metadata, progress=progress)
        synchronize()
        loading_seconds = time.perf_counter() - started
        s, a, metrics, history = fit_atlas(atlas, initial_s, alpha, q, run_id, bar, identity, state)
        del state
        del atlas  # Free expression memory before exporting the audited activities.
        exported = write_fit_h5(FIT_PATH, s, a, torch.from_numpy(initial_s), metadata,
                                config, provenance, metrics, run_id, BLOCK_ROWS, progress)
        final = history[-1]
        final.update(status='complete', loading_seconds=loading_seconds,
                     total_seconds=time.perf_counter() - started + resumed_seconds,
                     activity_nnz=exported['activity_nnz'])
        save_history(history)
        CHECKPOINT_PATH.unlink(missing_ok=True)
    # All cells were used for training; this error is not held-out prediction E.
    table = pd.DataFrame([{
        'K': len(initial_s), 'lambda': alpha, 'q': q, 'Cells': len(a), 'Updates': N_UPDATES,
        'Training objective / cell': final['training_objective'],
        'Relative reconstruction error': final['relative_reconstruction_error'],
        'Active atoms / cell': final['mean_active_atoms'], 'Used atoms': final['used_atoms'],
        'Total hours': final['total_seconds'] / 3600,
        'Peak GPU GiB': final['peak_gpu_gib'],
    }])
    display(table.style.format({'lambda': '{:.3g}', 'q': '{:.3g}',
                                'Training objective / cell': '{:.6f}',
                                'Relative reconstruction error': '{:.6f}',
                                'Active atoms / cell': '{:.2f}', 'Total hours': '{:.2f}',
                                'Peak GPU GiB': '{:.2f}'}).hide(axis='index')
            .set_caption(f'Verified outputs: {FIT_PATH} and {CSV_PATH}'))


run()